# MVO 核心代码：逐个单元格阅读与修改

所有项目算法集中在这个 Notebook 中。每个函数或类单独占一个代码单元格，前面的中文说明解释其作用。

从上到下运行只会安装依赖、导入库并定义函数；完整训练与回测在 [mvo.example.ipynb](mvo.example.ipynb) 中执行。

修改这里的函数并保存后，在 API、实验或测试 Notebook 中选择 **Restart Kernel and Run All Cells**，开头的 `%run ./mvo_utils.ipynb` 会载入修改后的定义。

阅读顺序：优化器 → 数据 → 训练 → 指标 → 近似网络 → 回测 → 报告。

## 0. 安装依赖与导入库

额外依赖固定版本安装到当前 Notebook 的 Python 环境，Dockerfile 和 requirements.txt 保持课程模板不变。首次安装若提示重启内核，按提示重启后从头运行。

In [1]:
%pip install -q scikit-learn==1.8.0 torch==2.8.0 cvxpy==1.9.2 cvxpylayers==1.2.0

Note: you may need to restart the kernel to use updated packages.


In [2]:
import cvxpy as cp
import numpy as np
from io import BytesIO, StringIO
from pathlib import Path
from urllib.request import urlopen
from zipfile import ZipFile
import hashlib
import json
import re
import pandas as pd
from sklearn.covariance import LedoitWolf
from copy import deepcopy
from dataclasses import dataclass
from time import perf_counter
import torch
from torch import nn
import math
from dataclasses import asdict, dataclass
import shutil

## 1. 均值—方差优化器

定义目标、非负满仓约束，以及可选的换手上限。普通求解器和可微层共用同一个问题。

### `covariance_factor`

协方差因子：检查矩阵，再构造满足 FᵀF = Σ 的因子。

In [3]:
def covariance_factor(covariance):
    """Return F with F.T @ F = covariance, allowing roundoff at zero."""
    covariance = np.asarray(covariance, dtype=float)
    if covariance.ndim != 2 or covariance.shape[0] != covariance.shape[1]:
        raise ValueError("Covariance must be a square matrix.")
    if not np.isfinite(covariance).all():
        raise ValueError("Covariance must be finite.")
    if not np.allclose(covariance, covariance.T, atol=1e-10, rtol=1e-8):
        raise ValueError("Covariance must be symmetric.")
    values, vectors = np.linalg.eigh((covariance + covariance.T) / 2)
    if values.min() < -1e-10:
        raise ValueError("Covariance must be positive semidefinite.")
    return np.sqrt(np.maximum(values, 0))[:, None] * vectors.T

### `clean_weights`

权重检查：只在误差处于求解器容差内时清理微小负值。

In [4]:
def clean_weights(weights, tolerance=2e-4):
    """Remove solver roundoff only after checking the original solution."""
    weights = np.asarray(weights, dtype=float)
    if weights.ndim != 1 or not np.isfinite(weights).all():
        raise ValueError("Portfolio weights must be a finite vector.")
    if weights.min() < -tolerance or abs(weights.sum() - 1) > tolerance:
        raise ValueError("Solver returned infeasible portfolio weights.")
    weights = np.maximum(weights, 0)
    return weights / weights.sum()

### `_allocation_problem`

统一问题定义：风险惩罚减去预期收益，附加预算与可选换手约束。

In [5]:
def _allocation_problem(n_assets, risk_aversion, turnover_limit=None):
    if n_assets < 2 or not np.isfinite(risk_aversion) or risk_aversion <= 0:
        raise ValueError("Use at least two assets and positive risk aversion.")
    mean = cp.Parameter(n_assets)
    factor = cp.Parameter((n_assets, n_assets))
    weights = cp.Variable(n_assets)
    previous = cp.Parameter(n_assets)
    # A covariance factor makes the problem DPP-compatible for differentiation.
    loss = risk_aversion * cp.sum_squares(factor @ weights) - mean @ weights
    constraints = [weights >= 0, cp.sum(weights) == 1]
    if turnover_limit is not None:
        if not np.isfinite(turnover_limit) or not 0 < turnover_limit <= 2:
            raise ValueError("Turnover limit must be in (0, 2].")
        constraints.append(cp.norm1(weights - previous) <= turnover_limit)
    problem = cp.Problem(cp.Minimize(loss), constraints)
    return problem, mean, factor, weights, previous

### `MeanVarianceSolver`

普通求解器：复用已定义的问题，对不同收益和协方差求最优权重。

In [6]:
class MeanVarianceSolver:
    """Reuse one compiled convex problem across annualized input moments."""

    def __init__(self, n_assets, risk_aversion=5.0, turnover_limit=None):
        self.problem, self.mean, self.factor, self.weights, self.previous = _allocation_problem(
            n_assets, risk_aversion, turnover_limit
        )
        self.turnover_limit = turnover_limit

    def solve(self, mean, covariance, previous=None):
        mean = np.asarray(mean, dtype=float)
        if mean.shape != self.mean.shape or not np.isfinite(mean).all():
            raise ValueError("Expected returns have an invalid shape or value.")
        self.mean.value = mean
        self.factor.value = covariance_factor(covariance)
        if self.turnover_limit is not None:
            if previous is None:
                raise ValueError("Provide previous invested weights for a turnover constraint.")
            previous = clean_weights(previous)
            self.previous.value = previous
        self.problem.solve(solver=cp.CLARABEL, warm_start=True)
        if self.problem.status not in (cp.OPTIMAL, cp.OPTIMAL_INACCURATE):
            raise RuntimeError(f"Allocation failed: {self.problem.status}")
        weights = clean_weights(self.weights.value)
        if self.turnover_limit is not None:
            if np.abs(weights - previous).sum() > self.turnover_limit + 2e-4:
                raise RuntimeError("Solver exceeded the turnover constraint.")
        return weights

### `make_allocation_layer`

可微优化层：让梯度穿过优化问题，传回预测网络。

In [7]:
def make_allocation_layer(n_assets, risk_aversion=5.0, turnover_limit=None):
    """Create a PyTorch layer accepting (annual_mean, covariance_factor)."""
    from cvxpylayers.torch import CvxpyLayer

    problem, mean, factor, weights, previous = _allocation_problem(
        n_assets, risk_aversion, turnover_limit
    )
    parameters = [mean, factor]
    if turnover_limit is not None:
        parameters.append(previous)
    return CvxpyLayer(problem, parameters=parameters, variables=[weights])

## 2. 数据与历史特征

读取真实行业收益，或生成测试用模拟数据。特征仅使用配置日之前的观测，未来收益只作为训练标签和评价数据。

In [8]:
FRENCH_URL = (
    "https://mba.tuck.dartmouth.edu/pages/faculty/Ken.French/ftp/"
    "10_Industry_Portfolios_daily_CSV.zip"
)

### `validate_returns`

输入校验：日期必须有序且唯一，收益必须是有效的小数。

In [9]:
def validate_returns(frame):
    frame = frame.copy()
    frame.index = pd.to_datetime(frame.index)
    if frame.empty or frame.shape[1] < 2:
        raise ValueError("Provide at least two assets and nonempty daily returns.")
    if frame.index.hasnans or frame.index.has_duplicates:
        raise ValueError("Dates must be valid and unique.")
    if not frame.index.is_monotonic_increasing or frame.columns.has_duplicates:
        raise ValueError("Dates must be sorted and asset names unique.")
    values = frame.to_numpy(dtype=float)
    if not np.isfinite(values).all() or np.any(values <= -1):
        raise ValueError("Returns must be finite decimals greater than -1.")
    frame.index.name = "date"
    return frame.astype(float)

### `parse_french`

解析数据：只取价值加权收益表，将百分数转换为小数。

In [10]:
def parse_french(payload):
    with ZipFile(BytesIO(payload)) as archive:
        text = archive.read(archive.namelist()[0]).decode("utf-8-sig")
    lines = text.splitlines()
    start = next(i for i, line in enumerate(lines) if "Average Value Weighted Returns" in line)
    header = next(i for i in range(start + 1, len(lines)) if lines[i].strip().startswith(","))
    rows = []
    for line in lines[header + 1:]:
        if not re.match(r"^\s*\d{8},", line):
            if rows:
                break
            continue
        rows.append(line)
    # The archive also contains equal-weighted returns. Read only its first table.
    frame = pd.read_csv(StringIO("\n".join([lines[header], *rows])), index_col=0)
    frame.index = pd.to_datetime(frame.index.astype(str), format="%Y%m%d")
    frame.columns = frame.columns.str.strip()
    frame = frame.replace([-99.99, -999.0], np.nan) / 100
    return frame

### `download_french`

下载与追溯：固定日期范围，并保存原始压缩包和 CSV 的哈希。

In [11]:
def download_french(destination, start="2010-01-01", end="2025-12-31"):
    with urlopen(FRENCH_URL, timeout=60) as response:
        payload = response.read()
    frame = validate_returns(parse_french(payload).loc[start:end])
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(destination)
    metadata = {"source": FRENCH_URL, "start": str(frame.index[0].date()),
                "end": str(frame.index[-1].date()), "rows": len(frame),
                "archive_sha256": hashlib.sha256(payload).hexdigest(),
                "csv_sha256": hashlib.sha256(destination.read_bytes()).hexdigest()}
    destination.with_suffix(".json").write_text(json.dumps(metadata, indent=2))
    return frame

### `synthetic_returns`

模拟数据：用于离线接口演示和测试，不替代真实市场实验。

In [12]:
def synthetic_returns(days=2200, assets=10, seed=7):
    rng = np.random.default_rng(seed)
    market = rng.normal(0.0002, 0.008, (days, 1))
    loading = rng.uniform(0.5, 1.2, (1, assets))
    noise = rng.normal(0, 0.008, (days, assets))
    values = market @ loading + noise + 0.0001
    return validate_returns(pd.DataFrame(values, index=pd.bdate_range("2010-01-01", periods=days),
                                         columns=[f"asset_{i}" for i in range(assets)]))

### `make_samples`

构造样本：历史窗口生成特征，随后互不重叠的持有期生成标签。

In [13]:
def make_samples(frame, lookback=126, horizon=21):
    if lookback < 63 or horizon < 2:
        raise ValueError("lookback must be >= 63 and horizon >= 2.")
    values = validate_returns(frame).to_numpy()
    starts = np.arange(lookback, len(values) - horizon + 1, horizon)
    features, factors, means, future = [], [], [], []
    for start in starts:
        # The allocation at start can only see returns strictly before start.
        history = values[start - lookback:start]
        feature = [history[-days:].mean(0) * 252 for days in (5, 21, 63)]
        features.append(np.concatenate([*feature, history.std(0) * np.sqrt(252)]))
        covariance = LedoitWolf().fit(history).covariance_ * 252
        factors.append(covariance_factor(covariance))
        means.append(history.mean(0) * 252)
        future.append(values[start:start + horizon])
    if not len(starts):
        raise ValueError("Insufficient observations for one complete feature/return window.")
    return {"features": np.array(features), "factors": np.array(factors),
            "means": np.array(means), "future": np.array(future), "starts": starts}

## 3. 预测网络与决策导向训练

使用同一个小网络比较预测误差和组合效用两种训练目标。标准化仅拟合训练集，验证集用于选择训练轮次。

### `MeanPredictor`

均值预测网络：输出限制在合理的年化尺度，减轻噪声放大。

In [14]:
class MeanPredictor(nn.Module):
    def __init__(self, n_features, n_assets):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(n_features, 32), nn.Tanh(),
            nn.Linear(32, n_assets), nn.Tanh(),
        )

    def forward(self, features):
        # 限制年化均值预测的尺度，减轻噪声预测引起的极端配置。
        return 0.25 * self.network(features)

### `FittedModel`

训练结果：保存网络与训练集标准化参数，用相同变换处理新样本。

In [15]:
@dataclass
class FittedModel:
    model: MeanPredictor
    center: np.ndarray
    scale: np.ndarray
    metadata: dict

    def predict_mean(self, features):
        values = np.asarray(features, dtype=float)
        if values.ndim not in (1, 2) or values.shape[-1] != len(self.center) or not np.isfinite(values).all():
            raise ValueError("Features must be finite and match the training dimension.")
        normalized = torch.as_tensor((values - self.center) / self.scale)
        self.model.eval()
        with torch.no_grad():
            return self.model(normalized).numpy()

### `buy_and_hold_returns`

可微持有收益：从资产累计财富计算组合每日收益。

In [16]:
def buy_and_hold_returns(weights, asset_returns):
    """初始权重确定后，持有期间随资产涨跌自然漂移。"""
    asset_wealth = torch.cumprod(1 + asset_returns, dim=1)
    wealth = torch.einsum("bhn,bn->bh", asset_wealth, weights)
    previous = torch.cat((torch.ones_like(wealth[:, :1]), wealth[:, :-1]), dim=1)
    return wealth / previous - 1

### `_loss`

训练目标：选择均值预测 MSE 或持有期均值—方差效用。

In [17]:
def _loss(model, batch, mode, layer, risk_aversion):
    features, factors, future = batch
    mean = model(features)
    if mode == "mse":
        return (mean - 252 * future.mean(dim=1)).square().mean()
    weights, = layer(mean, factors, solver_args={"eps": 1e-6, "max_iters": 10000})
    checked = weights.detach()
    if not torch.isfinite(checked).all():
        raise RuntimeError("Differentiable solver returned non-finite weights.")
    if checked.min() < -2e-4 or (checked.sum(dim=-1) - 1).abs().max() > 2e-4:
        raise RuntimeError("Differentiable solver returned infeasible weights.")
    # 清除求解器容差范围内的负权重，同时保留可微计算图。
    weights = weights.clamp_min(0)
    weights = weights / weights.sum(dim=-1, keepdim=True)
    returns = buy_and_hold_returns(weights, future)
    utility = 252 * (returns.mean(dim=1) - risk_aversion * returns.var(dim=1, unbiased=False))
    return -utility.mean()

### `_check_dataset`

样本校验：确保特征、协方差因子和未来收益的维度一致。

In [18]:
def _check_dataset(dataset, name):
    features, factors, future = (np.asarray(value, dtype=float) for value in dataset)
    if features.ndim != 2 or factors.ndim != 3 or future.ndim != 3:
        raise ValueError(f"Invalid {name} dimensions.")
    count, n_assets = len(features), future.shape[-1]
    if count == 0 or len(future) != count or factors.shape != (count, n_assets, n_assets):
        raise ValueError(f"Inconsistent or empty {name} samples.")
    if future.shape[1] < 2 or n_assets < 2:
        raise ValueError("Training requires at least two assets and two holding days.")
    if not all(np.isfinite(value).all() for value in (features, factors, future)):
        raise ValueError(f"Non-finite values in {name}.")
    if np.any(future <= -1):
        raise ValueError("Daily simple returns must exceed -1.")
    return features, factors, future

### `_tensor_dataset`

张量转换：使用训练集参数标准化特征，保持双精度优化计算。

In [19]:
def _tensor_dataset(dataset, center, scale):
    features, factors, future = dataset
    return tuple(torch.as_tensor(value, dtype=torch.float64) for value in (
        (features - center) / scale, factors, future,
    ))

### `_validation_loss`

验证损失：只计算评分，不更新网络权重。

In [20]:
def _validation_loss(model, dataset, mode, layer, risk_aversion, batch_size):
    total = 0.0
    model.eval()
    with torch.no_grad():
        for start in range(0, len(dataset[0]), batch_size):
            batch = tuple(value[start:start + batch_size] for value in dataset)
            loss = _loss(model, batch, mode, layer, risk_aversion)
            total += float(loss) * len(batch[0])
    return total / len(dataset[0])

### `_fit_epochs`

训练循环：更新参数，并保留验证损失最低的模型。

In [21]:
def _fit_epochs(model, train, validation, mode, layer, risk_aversion, epochs, batch_size):
    optimizer = torch.optim.Adam(model.parameters(), lr=0.003)
    best_state, best_loss, best_epoch = None, float("inf"), 0
    for epoch in range(epochs):
        model.train()
        for indices in torch.randperm(len(train[0])).split(batch_size):
            batch = tuple(value[indices] for value in train)
            optimizer.zero_grad()
            loss = _loss(model, batch, mode, layer, risk_aversion)
            if not torch.isfinite(loss):
                raise RuntimeError("Training produced a non-finite loss.")
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0, error_if_nonfinite=True)
            optimizer.step()
        score = _validation_loss(model, validation, mode, layer, risk_aversion, batch_size)
        if not np.isfinite(score):
            raise RuntimeError("Validation produced a non-finite loss.")
        if score < best_loss:
            best_state, best_loss, best_epoch = deepcopy(model.state_dict()), score, epoch + 1
    model.load_state_dict(best_state)
    return best_loss, best_epoch

### `train_model`

训练入口：先分开检查训练和验证数据，再拟合标准化与网络。

In [22]:
def train_model(train, validation, mode, risk_aversion=5.0, epochs=15, seed=7, batch_size=32):
    """验证集只用于选取训练轮次，标准化参数完全来自训练集。"""
    if mode not in {"mse", "utility"} or epochs < 1 or batch_size < 1:
        raise ValueError("Invalid training mode or hyperparameters.")
    if not np.isfinite(risk_aversion) or risk_aversion <= 0:
        raise ValueError("Risk aversion must be finite and positive.")
    train = _check_dataset(train, "training")
    validation = _check_dataset(validation, "validation")
    if any(a.shape[1:] != b.shape[1:] for a, b in zip(train, validation)):
        raise ValueError("Training and validation dimensions must match.")
    started = perf_counter()
    torch.manual_seed(seed)
    center = train[0].mean(axis=0)
    scale = train[0].std(axis=0)
    scale = np.where(scale > 1e-8, scale, 1.0)
    train_tensors = _tensor_dataset(train, center, scale)
    validation_tensors = _tensor_dataset(validation, center, scale)
    model = MeanPredictor(train[0].shape[-1], train[2].shape[-1]).double()
    layer = None
    if mode == "utility":
        layer = make_allocation_layer(train[2].shape[-1], risk_aversion)
    best_loss, best_epoch = _fit_epochs(
        model, train_tensors, validation_tensors, mode, layer, risk_aversion, epochs, batch_size,
    )
    metadata = {
        "mode": mode, "seed": seed, "epochs": epochs, "best_epoch": best_epoch,
        "validation_loss": best_loss, "train_seconds": perf_counter() - started,
        "training_samples": len(train[0]), "validation_samples": len(validation[0]),
    }
    return FittedModel(model.eval(), center, scale, metadata)

## 4. 持仓记账与评价指标

持有期间权重随价格自然漂移；调仓时按买卖金额扣费。配对区块 bootstrap 用于检查样本外效用差异的不确定性。

### `hold_period`

费用与漂移：调仓前扣费，随后保持资产份额，更新每日收益和期末权重。

In [23]:
def hold_period(target, previous, returns, cost_bps):
    """Charge cost per dollar bought or sold, then let holdings drift."""
    if cost_bps < 0:
        raise ValueError("Trading cost must be nonnegative.")
    turnover = float(np.abs(target - previous).sum())
    cost = cost_bps * 1e-4 * turnover
    if cost >= 1:
        raise ValueError("Trading costs consume the entire portfolio.")
    holdings = np.asarray(target, dtype=float).copy()
    daily = []
    for row in returns:
        growth = float(holdings @ (1 + row))
        daily.append(growth - 1)
        holdings = holdings * (1 + row) / growth
    # Costs are paid before the first day's market move.
    daily[0] = (1 - cost) * (1 + daily[0]) - 1
    return np.array(daily), holdings, turnover

### `summarize`

结果指标：计算 CAGR、波动率、Sharpe、回撤、效用与换手。

In [24]:
def summarize(daily, turnover, risk_aversion=5.0):
    values = np.asarray(daily)
    wealth = np.r_[1.0, np.cumprod(1 + values)]
    variance = float(values.var(ddof=0)) * 252
    volatility = np.sqrt(variance)
    mean = float(values.mean()) * 252
    return {"cagr": float(wealth[-1] ** (252 / len(values)) - 1),
            "annual_volatility": float(volatility),
            "sharpe_rf_zero": float(mean / volatility) if volatility > 0 else None,
            "max_drawdown": float((wealth / np.maximum.accumulate(wealth) - 1).min()),
            "annual_utility": mean - risk_aversion * variance,
            "annual_turnover": float(np.sum(turnover) * 252 / len(values))}

### `paired_utility_interval`

不确定性：对相同日期的策略收益成对抽取连续区块。

In [25]:
def paired_utility_interval(candidate, baseline, risk_aversion=5.0, seed=7,
                            block=21, draws=1000):
    """Circular block bootstrap keeps paired dates and short-run dependence."""
    rng = np.random.default_rng(seed)
    count = len(candidate)
    differences = []
    for _ in range(draws):
        starts = rng.integers(0, count, size=int(np.ceil(count / block)))
        indices = ((starts[:, None] + np.arange(block)) % count).ravel()[:count]
        left, right = np.asarray(candidate)[indices], np.asarray(baseline)[indices]
        delta = left.mean() - right.mean() - risk_aversion * (left.var() - right.var())
        differences.append(252 * delta)
    return {"lower_95": float(np.quantile(differences, 0.025)),
            "upper_95": float(np.quantile(differences, 0.975)),
            "block_days": block, "draws": draws}

## 5. 直接近似最优权重

在独立生成的训练与测试实例上，让神经网络学习求解器给出的权重，再比较误差和计算时间。

### `sample_instances`

优化实例：独立生成均值和正定协方差，不把模拟实例当作市场预测。

In [26]:
def sample_instances(count, n_assets, rng):
    """Independent annualized instances with positive definite covariance."""
    loadings = rng.normal(size=(count, n_assets, n_assets))
    covariance = loadings @ loadings.transpose(0, 2, 1) / n_assets
    covariance += np.eye(n_assets)[None] * 0.2
    scale = np.sqrt(np.diagonal(covariance, axis1=1, axis2=2))
    covariance /= scale[:, :, None] * scale[:, None, :]
    volatility = rng.uniform(0.1, 0.35, (count, n_assets))
    covariance *= volatility[:, :, None] * volatility[:, None, :]
    means = rng.uniform(-0.05, 0.2, (count, n_assets))
    return means, covariance

### `features`

网络输入：拼接均值、协方差和上一期权重。

In [27]:
def features(means, covariance, previous):
    return np.concatenate([means, covariance.reshape(len(means), -1), previous], axis=1)

### `allocation_model`

权重网络：Softmax 使输出非负且权重之和为一。

In [28]:
def allocation_model(n_assets):
    return nn.Sequential(
        nn.Linear(2 * n_assets + n_assets**2, 64),
        nn.ReLU(),
        nn.Linear(64, 64),
        nn.ReLU(),
        nn.Linear(64, n_assets),
        nn.Softmax(dim=-1),
    )

### `constrain_turnover`

换手上限：沿旧权重到建议权重的线段缩放，保持组合可行。

In [29]:
def constrain_turnover(weights, previous, limit):
    """Move along a simplex segment; its L1 length cannot exceed the limit."""
    if limit is None:
        return weights
    distance = (weights - previous).abs().sum(dim=-1, keepdim=True)
    fraction = (limit / distance.clamp_min(1e-12)).clamp(max=1)
    return previous + fraction * (weights - previous)

### `fit_model`

模仿求解器：用最优权重标签训练网络，标准化仅使用训练实例。

In [30]:
def fit_model(inputs, targets, n_assets, epochs, turnover_limit):
    center = inputs.mean(axis=0)
    scale = np.maximum(inputs.std(axis=0), 1e-8)
    x = torch.tensor((inputs - center) / scale, dtype=torch.float32)
    y = torch.tensor(targets, dtype=torch.float32)
    model = allocation_model(n_assets)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.003)
    model.train()
    for _ in range(epochs):
        optimizer.zero_grad()
        previous = torch.tensor(inputs[:, -n_assets:], dtype=torch.float32)
        weights = constrain_turnover(model(x), previous, turnover_limit)
        loss = (weights - y).square().mean()
        loss.backward()
        optimizer.step()
    model.eval()
    return model, center, scale

### `predict`

预测权重：输入整理与标准化计入推理时间。

In [31]:
def predict(model, center, scale, means, covariance, previous, turnover_limit):
    # Include input packing and normalization in inference timing.
    inputs = (features(means, covariance, previous) - center) / scale
    with torch.no_grad():
        weights = model(torch.tensor(inputs, dtype=torch.float32))
        prior = torch.tensor(previous, dtype=torch.float32)
        return constrain_turnover(weights, prior, turnover_limit).numpy()

### `benchmark`

计时：分别测量预热后的单实例延迟和网络批量吞吐量。

In [32]:
def benchmark(model, center, scale, means, covariance, previous, solver):
    predict(model, center, scale, means[:1], covariance[:1], previous[:1], solver.turnover_limit)
    solver.solve(means[0], covariance[0], previous[0])
    neural_times, solver_times = [], []
    for mean, cov, prior in zip(means, covariance, previous):
        start = perf_counter()
        predict(model, center, scale, mean[None], cov[None], prior[None], solver.turnover_limit)
        neural_times.append(perf_counter() - start)
        start = perf_counter()
        solver.solve(mean, cov, prior)
        solver_times.append(perf_counter() - start)
    batch_times = []
    for _ in range(20):
        start = perf_counter()
        predict(model, center, scale, means, covariance, previous, solver.turnover_limit)
        batch_times.append(perf_counter() - start)
    return {
        "nn_single_median_ms": float(np.median(neural_times) * 1000),
        "solver_single_median_ms": float(np.median(solver_times) * 1000),
        "nn_batch_instances_per_second": float(len(means) / np.median(batch_times)),
        "timing_device": "CPU",
        "torch_threads": torch.get_num_threads(),
    }

### `evaluate`

近似质量：比较目标函数差距、权重误差和预算约束误差。

In [33]:
def evaluate(weights, optimum, means, covariance, risk_aversion):
    def objective(portfolios):
        risk = np.einsum("bi,bij,bj->b", portfolios, covariance, portfolios)
        return risk_aversion * risk - np.sum(means * portfolios, axis=1)

    gap = objective(weights) - objective(optimum)
    equal = np.full_like(weights, 1 / weights.shape[1])
    return {
        "objective_gap_mean": float(gap.mean()),
        "objective_gap_p95": float(np.quantile(gap, 0.95)),
        "objective_gap_max": float(gap.max()),
        "equal_weight_objective_gap_mean": float((objective(equal) - objective(optimum)).mean()),
        "weight_rmse": float(np.sqrt(np.mean((weights - optimum) ** 2))),
        "minimum_weight": float(weights.min()),
        "maximum_budget_error": float(np.max(abs(weights.sum(axis=1) - 1))),
    }

### `run_experiment`

实验入口：生成独立训练和测试实例，求标签、训练，再评价。

In [34]:
def run_experiment(output_dir, seed=7, n_assets=10, train_size=512,
                   test_size=128, epochs=150, turnover_limit=None, risk_aversion=5.0):
    if min(train_size, test_size, epochs) < 1 or n_assets < 2:
        raise ValueError("Use positive sample/epoch counts and at least two assets.")
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)
    train_mean, train_cov = sample_instances(train_size, n_assets, rng)
    test_mean, test_cov = sample_instances(test_size, n_assets, rng)
    train_previous = rng.dirichlet(np.ones(n_assets), train_size)
    test_previous = rng.dirichlet(np.ones(n_assets), test_size)
    solver = MeanVarianceSolver(n_assets, risk_aversion, turnover_limit)
    start = perf_counter()
    targets = np.array([solver.solve(m, c, p)
                        for m, c, p in zip(train_mean, train_cov, train_previous)])
    label_seconds = perf_counter() - start
    start = perf_counter()
    model, center, scale = fit_model(
        features(train_mean, train_cov, train_previous), targets, n_assets, epochs, turnover_limit
    )
    train_seconds = perf_counter() - start
    return _save_experiment(output_dir, model, center, scale, solver,
                            test_mean, test_cov, test_previous, label_seconds, train_seconds,
                            seed, train_size, epochs, risk_aversion)

### `_save_experiment`

保存实验：记录误差、约束、时间与模型；计算时间收支平衡点。

In [35]:
def _save_experiment(output_dir, model, center, scale, solver, means, covariance, previous,
                     label_seconds, train_seconds, seed, train_size, epochs, risk_aversion):
    optimum = np.array([solver.solve(m, c, p) for m, c, p in zip(means, covariance, previous)])
    weights = predict(model, center, scale, means, covariance, previous, solver.turnover_limit)
    metrics = evaluate(weights, optimum, means, covariance, risk_aversion)
    metrics.update(benchmark(model, center, scale, means, covariance, previous, solver))
    metrics["turnover_limit"] = solver.turnover_limit
    metrics["maximum_turnover"] = float(np.abs(weights - previous).sum(axis=1).max())
    if solver.turnover_limit is not None:
        # Equal weight can violate a turnover cap, so its gap is not a feasible baseline.
        metrics.pop("equal_weight_objective_gap_mean")
    saving = (metrics["solver_single_median_ms"] - metrics["nn_single_median_ms"]) / 1000
    metrics.update({
        "label_seconds": label_seconds, "train_seconds": train_seconds,
        "break_even_instances": math.ceil((label_seconds + train_seconds) / saving) if saving > 0 else None,
        "seed": seed, "n_assets": means.shape[1], "train_size": train_size,
        "test_size": len(means), "epochs": epochs, "risk_aversion": risk_aversion,
        "interpretation": "Break-even ignores approximation error; review objective gaps separately.",
    })
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    (output_dir / "amortized_metrics.json").write_text(json.dumps(metrics, indent=2) + "\n")
    torch.save({"state_dict": model.state_dict(), "center": torch.tensor(center),
                "scale": torch.tensor(scale), "n_assets": means.shape[1],
                "turnover_limit": solver.turnover_limit},
               output_dir / "amortized_model.pt")
    return metrics

## 6. 按时间滚动回测

严格分开训练、验证与测试日期，逐折比较六种策略。跨折保留持仓，确保成本记账连续。

### `Config`

回测设置：把窗口、轮次、风险偏好、成本与随机种子放在一起。

In [36]:
@dataclass
class Config:
    lookback: int = 126
    horizon: int = 21
    train_periods: int = 60
    validation_periods: int = 12
    test_periods: int = 12
    epochs: int = 15
    risk_aversion: float = 5.0
    cost_bps: float = 10.0
    seed: int = 7

### `training_data`

时间切片：按持有期索引提取训练或验证样本。

In [37]:
def training_data(samples, start, end):
    return tuple(samples[key][start:end] for key in ("features", "factors", "future"))

### `fit_fold`

每折训练：用相同数据分别训练预测型与决策导向模型。

In [38]:
def fit_fold(samples, first_test, config, output):
    validation_start = first_test - config.validation_periods
    train_start = validation_start - config.train_periods
    train = training_data(samples, train_start, validation_start)
    validation = training_data(samples, validation_start, first_test)
    fitted = {}
    for name, mode in (("predict_then_optimize", "mse"), ("decision_focused", "utility")):
        model = train_model(train, validation, mode, risk_aversion=config.risk_aversion,
                            epochs=config.epochs, seed=config.seed)
        fitted[name] = model
        torch.save({"state_dict": model.model.state_dict(),
                    "center": torch.tensor(model.center), "scale": torch.tensor(model.scale),
                    "metadata": model.metadata}, output / f"fold_{first_test}_{mode}.pt")
    return fitted

### `allocate`

六种策略：等权、最小方差、历史均值、均值收缩及两种学习方法。

In [39]:
def allocate(samples, index, models, solver):
    factor = samples["factors"][index]
    covariance = factor.T @ factor
    mean = samples["means"][index]
    weights = {"equal_weight": np.full(len(mean), 1 / len(mean)),
               "minimum_variance": solver.solve(np.zeros_like(mean), covariance),
               "historical_mvo": solver.solve(mean, covariance),
               "shrunk_mean_mvo": solver.solve(0.25 * mean, covariance)}
    for name, model in models.items():
        predicted = model.predict_mean(samples["features"][index])
        weights[name] = solver.solve(predicted, covariance)
    return weights

### `evaluate_fold`

测试期执行：生成调仓权重，扣除成本并记录每日收益。

In [40]:
def evaluate_fold(frame, samples, first, last, models, solver, previous, config):
    daily_rows, weight_rows = [], []
    for index in range(first, last):
        start = samples["starts"][index]
        dates = frame.index[start:start + config.horizon]
        for name, target in allocate(samples, index, models, solver).items():
            # A new strategy starts in cash. Subsequent trades use drifted weights.
            before = previous.get(name, np.zeros_like(target))
            daily, previous[name], turnover = hold_period(
                target, before, samples["future"][index], config.cost_bps)
            daily_rows.extend({"date": date, "strategy": name, "return": value}
                              for date, value in zip(dates, daily))
            row = {"date": dates[0], "strategy": name, "turnover": turnover}
            row.update(dict(zip(frame.columns, target)))
            weight_rows.append(row)
    return daily_rows, weight_rows

### `fold_metadata`

日期记录：保存各折训练、验证、测试范围以检查时间隔离。

In [41]:
def fold_metadata(frame, samples, first, last, config, models):
    starts = samples["starts"]
    train_start = first - config.validation_periods - config.train_periods
    validation_start = first - config.validation_periods
    return {"training_first_label": str(frame.index[starts[train_start]].date()),
            "training_last_label": str(frame.index[starts[validation_start] - 1].date()),
            "validation_first_label": str(frame.index[starts[validation_start]].date()),
            "validation_last_label": str(frame.index[starts[first] - 1].date()),
            "test_start": str(frame.index[starts[first]].date()),
            "test_end": str(frame.index[starts[last - 1] + config.horizon - 1].date()),
            "models": {name: model.metadata for name, model in models.items()}}

### `run_backtest`

回测入口：逐折训练与测试，跨折延续每个策略的持仓。

In [42]:
def run_backtest(frame, output_dir, config=None, source="synthetic"):
    config = config or Config()
    if min(config.train_periods, config.validation_periods, config.test_periods, config.epochs) < 1:
        raise ValueError("Training, validation, test and epoch counts must be positive.")
    if not np.isfinite(config.cost_bps) or config.cost_bps < 0:
        raise ValueError("Cost in bps must be finite and nonnegative.")
    samples = make_samples(frame, config.lookback, config.horizon)
    first_test = config.train_periods + config.validation_periods
    if first_test >= len(samples["starts"]):
        raise ValueError("Not enough complete periods for train, validation and test.")
    output = Path(output_dir)
    output.mkdir(parents=True, exist_ok=True)
    solver = MeanVarianceSolver(frame.shape[1], config.risk_aversion)
    previous, daily_rows, weight_rows, folds = {}, [], [], []
    for first in range(first_test, len(samples["starts"]), config.test_periods):
        last = min(first + config.test_periods, len(samples["starts"]))
        print(f"Training fold {len(folds) + 1}, test {frame.index[samples['starts'][first]].date()}", flush=True)
        models = fit_fold(samples, first, config, output)
        daily, weights = evaluate_fold(frame, samples, first, last, models, solver, previous, config)
        daily_rows.extend(daily)
        weight_rows.extend(weights)
        folds.append(fold_metadata(frame, samples, first, last, config, models))
    return save_results(frame, output, config, source, daily_rows, weight_rows, folds)

### `save_results`

保存回测：输出收益、权重、指标、配对区间和配置。

In [43]:
def save_results(frame, output, config, source, daily_rows, weight_rows, folds):
    daily = pd.DataFrame(daily_rows).pivot(index="date", columns="strategy", values="return")
    weights = pd.DataFrame(weight_rows)
    metrics = {name: summarize(daily[name], weights.loc[weights.strategy == name, "turnover"],
                               config.risk_aversion) for name in daily.columns}
    for name in daily.columns:
        holdings = weights.loc[weights.strategy == name, list(frame.columns)]
        metrics[name]["maximum_target_weight"] = float(holdings.to_numpy().max())
    intervals = {name: paired_utility_interval(daily.decision_focused, daily[name],
                                              config.risk_aversion, config.seed)
                 for name in ("equal_weight", "minimum_variance", "predict_then_optimize")}
    report = {"config": asdict(config), "source": source, "assets": list(frame.columns),
              "input_sha256": hashlib.sha256(frame.to_csv().encode()).hexdigest(),
              "test_start": str(daily.index[0].date()), "test_end": str(daily.index[-1].date()),
              "test_days": len(daily), "metrics": metrics,
              "decision_focused_minus_baseline_utility_ci": intervals, "folds": folds}
    daily.to_csv(output / "daily_returns.csv")
    weights.to_csv(output / "weights.csv", index=False)
    pd.DataFrame(metrics).T.to_csv(output / "summary.csv", index_label="strategy")
    (output / "backtest_report.json").write_text(json.dumps(report, indent=2) + "\n")
    plot_results(daily, output)
    return report

### `plot_results`

画图：净值和回撤均包含初始资金，图片保存到实验目录。

In [44]:
def plot_results(daily, output):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    wealth = pd.concat([pd.DataFrame(1., index=[daily.index[0] - pd.Timedelta(days=1)],
                                    columns=daily.columns), (1 + daily).cumprod()])
    fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
    wealth.plot(ax=axes[0], ylabel="Net wealth (initial = 1)")
    (wealth / wealth.cummax() - 1).plot(ax=axes[1], ylabel="Drawdown", legend=False)
    fig.tight_layout()
    fig.savefig(output / "backtest.png", dpi=160)
    plt.close(fig)

## 7. 生成结果报告

从本次实验输出自动生成 Markdown 报告，保证文档中的表格与结果快照一致。

### `amortized_section`

计算实验表：展示近似误差、推理时间与训练成本。

In [45]:
def amortized_section(results):
    lines = ["## 实验 A：近似误差和计算时间", "",
             "| 设置 | 平均目标差距 | 权重 RMSE | 网络单次 ms | 求解器单次 ms | 收支平衡次数 |",
             "|---|---:|---:|---:|---:|---:|"]
    for name, metrics in results.items():
        count = metrics["break_even_instances"]
        count = str(count) if count is not None else "无（单次未加速）"
        lines.append(f"| {name} | {metrics['objective_gap_mean']:.5f} | "
                     f"{metrics['weight_rmse']:.5f} | {metrics['nn_single_median_ms']:.4f} | "
                     f"{metrics['solver_single_median_ms']:.4f} | {count} |")
    lines += ["", "时间受硬件和系统负载影响。收支平衡次数只比较计算成本，忽略近似误差；"
              "网络批量吞吐量不能直接当作单次延迟。不同约束的目标差距也不能直接互相排名。", ""]
    return lines

### `backtest_section`

策略比较表：汇总预先指定的种子，并如实报告比较结果。

In [46]:
def backtest_section(reports):
    strategies = next(iter(reports.values()))["metrics"]
    lines = ["## 实验 B：三个预先指定种子的均值", "",
             "| 策略 | CAGR | 波动率 | Sharpe | 最大回撤 | 净效用 | 年化换手 | 最大目标权重 |",
             "|---|---:|---:|---:|---:|---:|---:|---:|"]
    averages = {}
    for strategy in strategies:
        values = [report["metrics"][strategy] for report in reports.values()]
        mean = {key: sum(value[key] for value in values) / len(values) for key in values[0]}
        averages[strategy] = mean
        lines.append(f"| {strategy} | {mean['cagr']:.2%} | {mean['annual_volatility']:.2%} | "
                     f"{mean['sharpe_rf_zero']:.3f} | {mean['max_drawdown']:.2%} | "
                     f"{mean['annual_utility']:.4f} | {mean['annual_turnover']:.2f} | "
                     f"{mean['maximum_target_weight']:.2%} |")
    best = max(averages, key=lambda name: averages[name]["annual_utility"])
    lines += ["", f"本次平均净效用最高的策略是 `{best}`。", ""]
    for baseline in ("equal_weight", "minimum_variance", "predict_then_optimize", "shrunk_mean_mvo"):
        wins = sum(report["metrics"]["decision_focused"]["annual_utility"] >
                   report["metrics"][baseline]["annual_utility"] for report in reports.values())
        lines.append(f"- 决策导向净效用高于 `{baseline}`：{wins}/{len(reports)} 个种子。")
    return lines + ["", "种子共享市场历史，不能作为独立市场样本。收缩对照不能严格分解神经网络提升的因果来源。", ""]

### `uncertainty_section`

区间解释：检查效用差异区间是否跨零。

In [47]:
def uncertainty_section(report):
    intervals = report["decision_focused_minus_baseline_utility_ci"]
    lines = ["## 不确定性（固定 seed=7）", "",
             "| 决策导向减去基线 | 95% 区块 bootstrap 下界 | 上界 |",
             "|---|---:|---:|"]
    for name, interval in intervals.items():
        lines.append(f"| {name} | {interval['lower_95']:.4f} | {interval['upper_95']:.4f} |")
    if all(interval["lower_95"] <= 0 <= interval["upper_95"] for interval in intervals.values()):
        lines += ["", "所有区间跨零，本次检查未显示稳健的决策导向优势。"]
    return lines + ["", "区块 bootstrap 不能消除市场非平稳性和模型选择偏差。", ""]

### `write_report`

报告入口：组合各张表与局限说明，并复制固定种子的图。

In [48]:
def write_report(snapshot, root):
    root = Path(root)
    report = snapshot["backtests"]["7"]
    data = snapshot["data"]
    lines = ["# 实际运行结果与解释", "",
             "数值由完整实验自动生成。配置、哈希与逐折结果见 `experiment_snapshot.json`；"
             "测试环境和执行记录见 `validation.json`。", "",
             f"输入：{data['start']} 至 {data['end']}，{data['rows']} 个交易日、10 个行业组合。",
             f"样本外：{report['test_start']} 至 {report['test_end']}，"
             f"{report['test_days']} 个交易日、{len(report['folds'])} 折。"
             "末尾不足完整持有期的日期不进入评价。", ""]
    lines += amortized_section(snapshot["amortized"])
    lines += backtest_section(snapshot["backtests"])
    lines += uncertainty_section(report)
    lines += ["![固定种子7的净值和回撤](backtest_seed7.png)", "", "## 局限", "",
              "- 训练采用毛效用，评价采用扣费后净收益；二者并不完全一致。",
              "- 没有整数手数/持仓数量限制，也没有另设完全未触碰的最终市场测试期。",
              "- 行业组合是研究序列，存在历史修订，不能等同于可交易 ETF 的实盘回报。",
              "- CVXPYlayers/NumPy 接口出现上游弃用警告；当前数值和梯度检查通过。", ""]
    (root / "docs/experiment_report.md").write_text("\n".join(lines))
    shutil.copyfile(root / "results/french_seed7/backtest.png", root / "docs/backtest_seed7.png")